In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
from math import sqrt, erf

In [3]:
ab = pd.read_csv(
    r"C:\Users\sridh\OneDrive\Desktop\resume_Da projects\ab test\ab_data.csv"
)

countries = pd.read_csv(
    r"C:\Users\sridh\OneDrive\Desktop\resume_Da projects\ab test\countries.csv"
)

In [7]:
# Merge datasets

df = ab.merge(
    countries,
    on='user_id',
    how='left'
)

print(df.shape)
df.head()

(294478, 6)


,user_id,timestamp,group,landing_page,converted,country
0,851104,2017-01-21 22:11:48.556739,control,old_page,0,US
1,804228,2017-01-12 08:01:45.159739,control,old_page,0,US
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0,US
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0,US
4,864975,2017-01-21 01:52:26.210827,control,old_page,1,US


In [8]:
import sqlite3

conn = sqlite3.connect("ab_testing.db")

df.to_sql(
    "ab_test",
    conn,
    if_exists="replace",
    index=False
)

print("Table created successfully")

Table created successfully


In [9]:
query = """
SELECT
    COUNT(*) AS total_users,
    SUM(converted) AS total_conversions,
    ROUND(
        100.0 * SUM(converted) / COUNT(*),
        2
    ) AS conversion_rate
FROM ab_test;
"""

pd.read_sql(query, conn)

,total_users,total_conversions,conversion_rate
0,294478,35237,11.97


In [10]:
query = """
SELECT
    "group",
    COUNT(*) AS users,
    SUM(converted) AS conversions,
    ROUND(
        100.0 * SUM(converted) / COUNT(*),
        2
    ) AS conversion_rate
FROM ab_test
GROUP BY "group";
"""

pd.read_sql(query, conn)

,group,users,conversions,conversion_rate
0,control,147202,17723,12.04
1,treatment,147276,17514,11.89


In [11]:
query = """
SELECT
    country,
    "group",
    COUNT(*) AS users,
    SUM(converted) AS conversions,
    ROUND(
        100.0 * SUM(converted) / COUNT(*),
        2
    ) AS conversion_rate
FROM ab_test
GROUP BY country, "group"
ORDER BY country;
"""

pd.read_sql(query, conn)

,country,group,users,conversions,conversion_rate
0,CA,control,7302,871,11.93
1,CA,treatment,7393,832,11.25
2,UK,control,36841,4428,12.02
3,UK,treatment,36578,4425,12.10
4,US,control,103059,12424,12.06
5,US,treatment,103305,12257,11.86


In [12]:
AOV = 2000

query = f"""
SELECT
    "group",
    COUNT(*) AS users,
    SUM(converted) AS conversions,
    ROUND(
        SUM(converted) * {AOV},
        2
    ) AS estimated_revenue
FROM ab_test
GROUP BY "group";
"""

pd.read_sql(query, conn)

,group,users,conversions,estimated_revenue
0,control,147202,17723,35446000.0
1,treatment,147276,17514,35028000.0


In [13]:
query = """
SELECT
    country,
    ROUND(
        100.0 * AVG(converted),
        2
    ) AS conversion_rate
FROM ab_test
GROUP BY country
ORDER BY conversion_rate DESC;
"""

pd.read_sql(query, conn)

,country,conversion_rate
0,UK,12.06
1,US,11.96
2,CA,11.59


In [1]:
-- Overall Conversion Rate
SELECT
    group_name,
    COUNT(*) AS users,
    SUM(converted) AS conversions,
    ROUND(100.0 * SUM(converted) / COUNT(*), 2) AS conversion_rate
FROM ab_test
GROUP BY group_name;

SyntaxError: invalid syntax (4279534068.py, line 1)